# Agentic Coding | Domain Applications

In [1]:
import sys
import subprocess
import tempfile
import os
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, Literal
from typing_extensions import NotRequired
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
class CodingState(TypedDict):
    spec: str
    code: NotRequired[str]
    test_code: NotRequired[str]
    execution_result: NotRequired[str]
    iteration: NotRequired[int]
    final_code: NotRequired[str]

MAX_ITERATIONS = 4

def generate_code(state: CodingState) -> dict:
    """Generate or fix code based on spec and any previous errors."""
    if state.get("execution_result") and "Error" in state.get("execution_result", ""):
        prompt = (
            f"Fix the following Python code based on the error.\n\n"
            f"Specification: {state['spec']}\n\n"
            f"Current code:\n```python\n{state['code']}\n```\n\n"
            f"Error:\n{state['execution_result']}\n\n"
            f"Return ONLY the corrected Python code (no markdown fences, no explanation)."
        )
    else:
        prompt = (
            f"Write a Python function based on this specification:\n\n"
            f"{state['spec']}\n\n"
            f"Also write pytest-style test functions that thoroughly test it.\n"
            f"Return ONLY valid Python code (no markdown fences). "
            f"Include both the function and the tests in a single script. "
            f"End with: if __name__ == '__main__': test functions called directly."
        )
    response = model.invoke(prompt)
    # Strip markdown fences if present
    code = response.content.strip()
    if code.startswith("```python"):
        code = code[len("```python"):].strip()
    if code.startswith("```"):
        code = code[3:].strip()
    if code.endswith("```"):
        code = code[:-3].strip()
    return {"code": code, "iteration": state.get("iteration", 0) + 1}

def execute_code(state: CodingState) -> Command[Literal["generate_code", "finalize"]]:
    """Execute the generated code and route based on result."""
    code = state["code"]
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, dir=tempfile.gettempdir()) as f:
            f.write(code)
            tmp_path = f.name
        result = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True, text=True, timeout=15
        )
        os.unlink(tmp_path)
        if result.returncode != 0:
            exec_result = f"Error (exit code {result.returncode}):\n{result.stderr}"
        else:
            exec_result = result.stdout.strip() or "Code executed successfully (no output)."
    except subprocess.TimeoutExpired:
        exec_result = "Error: Code execution timed out (15s limit)."
    except Exception as e:
        exec_result = f"Error: {str(e)}"

    iteration = state.get("iteration", 0)
    if "Error" not in exec_result or iteration >= MAX_ITERATIONS:
        return Command(goto="finalize", update={"execution_result": exec_result})
    return Command(goto="generate_code", update={"execution_result": exec_result})

def finalize(state: CodingState) -> dict:
    return {"final_code": state["code"]}

In [5]:
# Build graph
graph = StateGraph(CodingState)
graph.add_node("generate_code", generate_code)
graph.add_node("execute_code", execute_code, destinations=("generate_code", "finalize"))
graph.add_node("finalize", finalize)

graph.add_edge(START, "generate_code")
graph.add_edge("generate_code", "execute_code")
graph.add_edge("finalize", END)

coder = graph.compile()

In [6]:
# Plot the workflow
plot_mermaid(coder)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	generate_code(generate_code)
	execute_code(execute_code)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> generate_code;
	execute_code -.-> finalize;
	execute_code -.-> generate_code;
	generate_code --> execute_code;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [7]:
result = coder.invoke({
    "spec": "Write a function 'merge_sorted_lists(list1, list2)' that merges two sorted lists into one sorted list. "
            "Handle edge cases: empty lists, lists of different lengths, duplicate values."
})
print(f"Iterations: {result['iteration']}")
print(f"Execution: {result['execution_result']}")
print(f"\nFinal Code:\n{result['final_code']}")

Iterations: 1
Execution: All tests passed!

Final Code:
def merge_sorted_lists(list1, list2):
    merged_list = []
    i, j = 0, 0

    # Merge the two lists
    while i < len(list1) and j < len(list2):
        if list1[i] <= list2[j]:
            merged_list.append(list1[i])
            i += 1
        else:
            merged_list.append(list2[j])
            j += 1

    # Append any remaining elements from list1 or list2
    while i < len(list1):
        merged_list.append(list1[i])
        i += 1
    while j < len(list2):
        merged_list.append(list2[j])
        j += 1

    return merged_list

def test_merge_sorted_lists_empty_lists():
    assert merge_sorted_lists([], []) == []
    assert merge_sorted_lists([], [1, 2, 3]) == [1, 2, 3]
    assert merge_sorted_lists([1, 2, 3], []) == [1, 2, 3]

def test_merge_sorted_lists_different_lengths():
    assert merge_sorted_lists([1, 5, 9], [2, 3]) == [1, 2, 3, 5, 9]
    assert merge_sorted_lists([1, 3], [2, 4, 5, 6]) == [1, 2, 3, 4, 5, 

In [8]:
stream_invoke(coder, {
    "spec": "Write a function 'merge_sorted_lists(list1, list2)' that merges two sorted lists into one sorted list. "
            "Handle edge cases: empty lists, lists of different lengths, duplicate values."
})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'spec': "Write a function 'merge_sorted_lists(list1, list2)' that merges two sorted lists into one sorted list. Handle edge cases: empty lists, lists of different lengths, duplicate values.",
 'code': 'def merge_sorted_lists(list1, list2):\n    merged_list = []\n    i, j = 0, 0\n\n    # Merge the two lists\n    while i < len(list1) and j < len(list2):\n        if list1[i] <= list2[j]:\n            merged_list.append(list1[i])\n            i += 1\n        else:\n            merged_list.append(list2[j])\n            j += 1\n\n    # Append any remaining elements from list1 or list2\n    while i < len(list1):\n        merged_list.append(list1[i])\n        i += 1\n    while j < len(list2):\n        merged_list.append(list2[j])\n        j += 1\n\n    return merged_list\n\ndef test_merge_sorted_lists_empty_lists():\n    assert merge_sorted_lists([], []) == []\n    assert merge_sorted_lists([], [1, 2, 3]) == [1, 2, 3]\n    assert merge_sorted_lists([1, 2, 3], []) == [1, 2, 3]\n\ndef test_merg